# VGC Decision Transformer — v2
Pipeline: maschera legale corretta (spazio azioni 360, validata sui replay), dataset numpy
vettorizzato con cache in RAM, data augmentation, dropout, Flash Attention (fix NaN da
padding), AMP + `nn.DataParallel` per 2 GPU, reward = return-to-go.

**Due modalita'** (cella di config):
- `FINETUNE = False` — addestramento normale da zero (cosine LR decay).
- `FINETUNE = True` — fine-tuning da pesi pre-addestrati (`PRETRAINED`): baseline
  zero-shot prima di partire, LR basso, warmup lineare, backbone congelato per le
  prime epoche (allena solo ultimo blocco + testa), poi fine-tuning completo.

`FAST = True` per un sanity check veloce.

In [ ]:
import numpy as np  # type: ignore
import torch  # type: ignore
import torch.nn as nn  # type: ignore
import torch.nn.functional as F  # type: ignore
from torch.optim import AdamW  # type: ignore
from torch.utils.data import Dataset, DataLoader  # type: ignore
import os, sys, time, glob, random, math

# ── CONFIG ───────────────────────────────────────────────
FAST = False          # True = sanity check: 300 partite, modello piccolo
FINETUNE = False      # True = fine-tuning da pesi pre-addestrati

EPOCHS = 100
BATCH = 64            # con DataParallel il batch e' diviso tra le GPU
LR = 1e-4
D_MODEL, DEPTH, N_HEADS = 256, 6, 8
DROPOUT = 0.15        # anti-overfit (dataset piccolo)
AUGMENT = True        # permutazione righe pokemon + ordine mosse (solo train)
MAX_TURN = 49
WARMUP_EPOCHS = 0
FREEZE_EPOCHS = 0

DATA_GLOBS = [
    '/kaggle/input/**/reg_m-A/*.npz',
    '../npz/reg_m-A/*.npz',      # esecuzione locale
]

if FINETUNE:
    # NB: l'architettura (D_MODEL/DEPTH/N_HEADS) deve combaciare con i pesi!
    PRETRAINED = '/kaggle/input/<dataset-pesi>/vgc_decision_transformer.pth'
    EPOCHS = 30
    LR = 1e-5             # si parte gia' vicino a una buona soluzione
    WARMUP_EPOCHS = 2     # warmup lineare: evita di distruggere i pesi all'inizio
    FREEZE_EPOCHS = 3     # prime epoche: solo ultimo blocco + testa
    DATA_GLOBS = [
        '/kaggle/input/**/reg_m-B/*.npz',
        '../npz/reg_m-B/*.npz',
    ]

if FAST:
    EPOCHS, D_MODEL, DEPTH, N_HEADS = 10, 128, 3, 4

# checkpoint separati per configurazione (evita resume con size mismatch)
_CKPT = f"checkpoints_d{D_MODEL}x{DEPTH}" + ('_ft' if FINETUNE else '')
SAVE_DIR = f'/kaggle/working/{_CKPT}' if os.path.isdir('/kaggle/working') else _CKPT


In [ ]:
# ── Mappe ID -> indice embedding ─────────────────────
"""Mappe ID -> indice embedding (estratte da Embedding.py).

Convenzione: indice 0 = sconosciuto/padding; gli ID noti sono mappati a 1..N.
Le LUT numpy permettono il remap vettoriale nel Dataset.
"""
import numpy as np  # type: ignore

ability_map = {
        "0": 0,
        "1": 1,
        "2": 2,
        "3": 3,
        "4": 4,
        "5": 5,
        "7": 6,
        "8": 7,
        "9": 8,
        "10": 9,
        "11": 10,
        "12": 11,
        "13": 12,
        "14": 13,
        "15": 14,
        "17": 15,
        "18": 16,
        "19": 17,
        "20": 18,
        "22": 19,
        "23": 20,
        "24": 21,
        "26": 22,
        "27": 23,
        "28": 24,
        "29": 25,
        "30": 26,
        "31": 27,
        "33": 28,
        "34": 29,
        "35": 30,
        "36": 31,
        "37": 32,
        "38": 33,
        "39": 34,
        "40": 35,
        "43": 36,
        "44": 37,
        "45": 38,
        "46": 39,
        "47": 40,
        "48": 41,
        "49": 42,
        "51": 43,
        "52": 44,
        "53": 45,
        "56": 46,
        "59": 47,
        "61": 48,
        "62": 49,
        "63": 50,
        "65": 51,
        "66": 52,
        "67": 53,
        "68": 54,
        "69": 55,
        "70": 56,
        "72": 57,
        "73": 58,
        "74": 59,
        "75": 60,
        "77": 61,
        "79": 62,
        "80": 63,
        "81": 64,
        "82": 65,
        "84": 66,
        "86": 67,
        "87": 68,
        "89": 69,
        "91": 70,
        "92": 71,
        "94": 72,
        "99": 73,
        "101": 74,
        "102": 75,
        "104": 76,
        "107": 77,
        "108": 78,
        "109": 79,
        "111": 80,
        "113": 81,
        "115": 82,
        "117": 83,
        "119": 84,
        "124": 85,
        "125": 86,
        "126": 87,
        "127": 88,
        "128": 89,
        "130": 90,
        "131": 91,
        "132": 92,
        "133": 93,
        "134": 94,
        "136": 95,
        "140": 96,
        "141": 97,
        "142": 98,
        "143": 99,
        "144": 100,
        "146": 101,
        "149": 102,
        "151": 103,
        "152": 104,
        "153": 105,
        "154": 106,
        "156": 107,
        "157": 108,
        "158": 109,
        "159": 110,
        "166": 111,
        "168": 112,
        "171": 113,
        "172": 114,
        "173": 115,
        "174": 116,
        "175": 117,
        "176": 118,
        "177": 119,
        "178": 120,
        "180": 121,
        "181": 122,
        "182": 123,
        "184": 124,
        "185": 125,
        "187": 126,
        "192": 127,
        "196": 128,
        "199": 129,
        "201": 130,
        "204": 131,
        "209": 132,
        "212": 133,
        "214": 134,
        "215": 135,
        "222": 136,
        "226": 137,
        "240": 138,
        "242": 139,
        "245": 140,
        "247": 141,
        "254": 142,
        "258": 143,
        "259": 144,
        "260": 145,
        "261": 146,
        "272": 147,
        "278": 148,
        "280": 149,
        "283": 150,
        "290": 151,
        "291": 152,
        "292": 153,
        "295": 154,
        "296": 155,
        "297": 156,
        "300": 157,
        "301": 158,
        "308": 159,
        "309": 160,
        "310": 161,
        "311": 162,
        "312": 163,
        "313": 164
    }

move_map = {
        "0": 0,
        "7": 1,
        "8": 2,
        "9": 3,
        "14": 4,
        "18": 5,
        "25": 6,
        "32": 7,
        "34": 8,
        "38": 9,
        "42": 10,
        "44": 11,
        "46": 12,
        "50": 13,
        "53": 14,
        "56": 15,
        "57": 16,
        "58": 17,
        "59": 18,
        "63": 19,
        "65": 20,
        "67": 21,
        "73": 22,
        "76": 23,
        "79": 24,
        "81": 25,
        "83": 26,
        "85": 27,
        "86": 28,
        "87": 29,
        "89": 30,
        "90": 31,
        "92": 32,
        "94": 33,
        "95": 34,
        "98": 35,
        "101": 36,
        "103": 37,
        "105": 38,
        "107": 39,
        "109": 40,
        "113": 41,
        "114": 42,
        "115": 43,
        "116": 44,
        "120": 45,
        "126": 46,
        "127": 47,
        "136": 48,
        "137": 49,
        "141": 50,
        "153": 51,
        "156": 52,
        "157": 53,
        "162": 54,
        "164": 55,
        "165": 56,
        "174": 57,
        "182": 58,
        "183": 59,
        "184": 60,
        "187": 61,
        "188": 62,
        "189": 63,
        "192": 64,
        "194": 65,
        "195": 66,
        "196": 67,
        "197": 68,
        "198": 69,
        "200": 70,
        "201": 71,
        "202": 72,
        "203": 73,
        "204": 74,
        "207": 75,
        "213": 76,
        "219": 77,
        "220": 78,
        "223": 79,
        "224": 80,
        "226": 81,
        "227": 82,
        "234": 83,
        "235": 84,
        "236": 85,
        "238": 86,
        "240": 87,
        "241": 88,
        "242": 89,
        "243": 90,
        "244": 91,
        "245": 92,
        "246": 93,
        "247": 94,
        "248": 95,
        "250": 96,
        "251": 97,
        "252": 98,
        "254": 99,
        "257": 100,
        "261": 101,
        "262": 102,
        "263": 103,
        "266": 104,
        "269": 105,
        "270": 106,
        "271": 107,
        "272": 108,
        "273": 109,
        "276": 110,
        "278": 111,
        "280": 112,
        "281": 113,
        "282": 114,
        "283": 115,
        "284": 116,
        "285": 117,
        "286": 118,
        "297": 119,
        "298": 120,
        "299": 121,
        "303": 122,
        "304": 123,
        "305": 124,
        "307": 125,
        "308": 126,
        "309": 127,
        "311": 128,
        "313": 129,
        "314": 130,
        "315": 131,
        "317": 132,
        "319": 133,
        "321": 134,
        "322": 135,
        "323": 136,
        "326": 137,
        "328": 138,
        "329": 139,
        "330": 140,
        "331": 141,
        "332": 142,
        "333": 143,
        "334": 144,
        "336": 145,
        "337": 146,
        "338": 147,
        "339": 148,
        "344": 149,
        "347": 150,
        "348": 151,
        "349": 152,
        "350": 153,
        "352": 154,
        "355": 155,
        "356": 156,
        "359": 157,
        "360": 158,
        "361": 159,
        "364": 160,
        "366": 161,
        "367": 162,
        "368": 163,
        "369": 164,
        "370": 165,
        "371": 166,
        "372": 167,
        "374": 168,
        "383": 169,
        "387": 170,
        "388": 171,
        "389": 172,
        "394": 173,
        "396": 174,
        "398": 175,
        "399": 176,
        "400": 177,
        "402": 178,
        "403": 179,
        "404": 180,
        "405": 181,
        "406": 182,
        "407": 183,
        "408": 184,
        "409": 185,
        "410": 186,
        "411": 187,
        "412": 188,
        "413": 189,
        "414": 190,
        "415": 191,
        "416": 192,
        "417": 193,
        "418": 194,
        "419": 195,
        "420": 196,
        "421": 197,
        "423": 198,
        "425": 199,
        "427": 200,
        "428": 201,
        "430": 202,
        "432": 203,
        "433": 204,
        "434": 205,
        "435": 206,
        "436": 207,
        "437": 208,
        "438": 209,
        "439": 210,
        "441": 211,
        "442": 212,
        "444": 213,
        "446": 214,
        "447": 215,
        "450": 216,
        "452": 217,
        "453": 218,
        "457": 219,
        "469": 220,
        "473": 221,
        "474": 222,
        "476": 223,
        "479": 224,
        "480": 225,
        "482": 226,
        "483": 227,
        "484": 228,
        "487": 229,
        "488": 230,
        "489": 231,
        "490": 232,
        "491": 233,
        "492": 234,
        "493": 235,
        "494": 236,
        "495": 237,
        "496": 238,
        "499": 239,
        "500": 240,
        "501": 241,
        "502": 242,
        "503": 243,
        "504": 244,
        "505": 245,
        "506": 246,
        "509": 247,
        "511": 248,
        "512": 249,
        "513": 250,
        "515": 251,
        "521": 252,
        "522": 253,
        "523": 254,
        "524": 255,
        "525": 256,
        "527": 257,
        "528": 258,
        "529": 259,
        "532": 260,
        "533": 261,
        "534": 262,
        "535": 263,
        "538": 264,
        "542": 265,
        "552": 266,
        "555": 267,
        "556": 268,
        "566": 269,
        "567": 270,
        "570": 271,
        "572": 272,
        "573": 273,
        "575": 274,
        "576": 275,
        "577": 276,
        "580": 277,
        "583": 278,
        "585": 279,
        "586": 280,
        "588": 281,
        "594": 282,
        "595": 283,
        "596": 284,
        "598": 285,
        "604": 286,
        "605": 287,
        "608": 288,
        "609": 289,
        "611": 290,
        "617": 291,
        "661": 292,
        "663": 293,
        "665": 294,
        "667": 295,
        "668": 296,
        "669": 297,
        "675": 298,
        "676": 299,
        "678": 300,
        "679": 301,
        "681": 302,
        "682": 303,
        "683": 304,
        "688": 305,
        "689": 306,
        "691": 307,
        "693": 308,
        "694": 309,
        "706": 310,
        "707": 311,
        "709": 312,
        "710": 313,
        "747": 314,
        "748": 315,
        "751": 316,
        "775": 317,
        "776": 318,
        "777": 319,
        "783": 320,
        "784": 321,
        "787": 322,
        "788": 323,
        "789": 324,
        "791": 325,
        "796": 326,
        "797": 327,
        "799": 328,
        "801": 329,
        "802": 330,
        "803": 331,
        "804": 332,
        "806": 333,
        "807": 334,
        "808": 335,
        "809": 336,
        "811": 337,
        "812": 338,
        "813": 339,
        "814": 340,
        "815": 341,
        "826": 342,
        "827": 343,
        "828": 344,
        "830": 345,
        "834": 346,
        "836": 347,
        "838": 348,
        "839": 349,
        "841": 350,
        "842": 351,
        "843": 352,
        "845": 353,
        "854": 354,
        "855": 355,
        "857": 356,
        "858": 357,
        "860": 358,
        "861": 359,
        "864": 360,
        "866": 361,
        "869": 362,
        "870": 363,
        "871": 364,
        "872": 365,
        "873": 366,
        "874": 367,
        "880": 368,
        "881": 369,
        "883": 370,
        "885": 371,
        "886": 372,
        "888": 373,
        "889": 374,
        "890": 375,
        "891": 376,
        "893": 377,
        "895": 378,
        "902": 379,
        "905": 380,
        "907": 381,
        "912": 382,
        "913": 383,
        "914": 384,
        "915": 385,
        "916": 386,
        "917": 387,
        "918": 388
    }

pokemon_map = {
        "0": 0,
        "3": 1,
        "6": 2,
        "9": 3,
        "15": 4,
        "18": 5,
        "25": 6,
        "26": 7,
        "36": 8,
        "38": 9,
        "45": 10,
        "59": 11,
        "65": 12,
        "68": 13,
        "71": 14,
        "80": 15,
        "94": 16,
        "115": 17,
        "121": 18,
        "127": 19,
        "128": 20,
        "130": 21,
        "132": 22,
        "134": 23,
        "135": 24,
        "136": 25,
        "142": 26,
        "143": 27,
        "149": 28,
        "154": 29,
        "157": 30,
        "160": 31,
        "168": 32,
        "181": 33,
        "184": 34,
        "186": 35,
        "196": 36,
        "197": 37,
        "199": 38,
        "205": 39,
        "208": 40,
        "211": 41,
        "212": 42,
        "214": 43,
        "227": 44,
        "229": 45,
        "248": 46,
        "254": 47,
        "257": 48,
        "260": 49,
        "279": 50,
        "282": 51,
        "302": 52,
        "303": 53,
        "306": 54,
        "308": 55,
        "310": 56,
        "319": 57,
        "323": 58,
        "324": 59,
        "334": 60,
        "350": 61,
        "351": 62,
        "354": 63,
        "358": 64,
        "359": 65,
        "362": 66,
        "376": 67,
        "389": 68,
        "392": 69,
        "395": 70,
        "398": 71,
        "405": 72,
        "407": 73,
        "409": 74,
        "411": 75,
        "428": 76,
        "442": 77,
        "445": 78,
        "448": 79,
        "450": 80,
        "454": 81,
        "460": 82,
        "461": 83,
        "464": 84,
        "470": 85,
        "471": 86,
        "472": 87,
        "473": 88,
        "475": 89,
        "478": 90,
        "497": 91,
        "500": 92,
        "503": 93,
        "510": 94,
        "516": 95,
        "518": 96,
        "530": 97,
        "531": 98,
        "534": 99,
        "545": 100,
        "547": 101,
        "553": 102,
        "560": 103,
        "563": 104,
        "569": 105,
        "571": 106,
        "579": 107,
        "584": 108,
        "604": 109,
        "609": 110,
        "614": 111,
        "623": 112,
        "635": 113,
        "637": 114,
        "652": 115,
        "655": 116,
        "658": 117,
        "660": 118,
        "663": 119,
        "666": 120,
        "668": 121,
        "671": 122,
        "675": 123,
        "678": 124,
        "681": 125,
        "683": 126,
        "687": 127,
        "689": 128,
        "691": 129,
        "693": 130,
        "695": 131,
        "697": 132,
        "699": 133,
        "700": 134,
        "701": 135,
        "706": 136,
        "707": 137,
        "709": 138,
        "711": 139,
        "715": 140,
        "727": 141,
        "730": 142,
        "733": 143,
        "740": 144,
        "745": 145,
        "748": 146,
        "750": 147,
        "752": 148,
        "758": 149,
        "763": 150,
        "765": 151,
        "766": 152,
        "778": 153,
        "780": 154,
        "784": 155,
        "823": 156,
        "841": 157,
        "842": 158,
        "844": 159,
        "855": 160,
        "858": 161,
        "861": 162,
        "866": 163,
        "867": 164,
        "869": 165,
        "870": 166,
        "877": 167,
        "887": 168,
        "899": 169,
        "900": 170,
        "902": 171,
        "903": 172,
        "904": 173,
        "908": 174,
        "911": 175,
        "914": 176,
        "925": 177,
        "934": 178,
        "936": 179,
        "937": 180,
        "939": 181,
        "952": 182,
        "956": 183,
        "959": 184,
        "964": 185,
        "968": 186,
        "970": 187,
        "972": 188,
        "979": 189,
        "981": 190,
        "983": 191,
        "1000": 192,
        "1013": 193,
        "1018": 194,
        "1019": 195,
        "10008": 196,
        "10009": 197,
        "10010": 198,
        "10011": 199,
        "10012": 200,
        "10032": 201,
        "10033": 202,
        "10034": 203,
        "10035": 204,
        "10036": 205,
        "10037": 206,
        "10038": 207,
        "10039": 208,
        "10040": 209,
        "10041": 210,
        "10042": 211,
        "10045": 212,
        "10046": 213,
        "10047": 214,
        "10048": 215,
        "10049": 216,
        "10050": 217,
        "10051": 218,
        "10052": 219,
        "10053": 220,
        "10054": 221,
        "10055": 222,
        "10056": 223,
        "10057": 224,
        "10058": 225,
        "10059": 226,
        "10060": 227,
        "10061": 228,
        "10064": 229,
        "10065": 230,
        "10066": 231,
        "10067": 232,
        "10068": 233,
        "10069": 234,
        "10070": 235,
        "10071": 236,
        "10072": 237,
        "10073": 238,
        "10074": 239,
        "10076": 240,
        "10087": 241,
        "10088": 242,
        "10090": 243,
        "10104": 244,
        "10126": 245,
        "10143": 246,
        "10152": 247,
        "10165": 248,
        "10172": 249,
        "10230": 250,
        "10233": 251,
        "10236": 252,
        "10239": 253,
        "10242": 254,
        "10243": 255,
        "10244": 256,
        "10248": 257,
        "10250": 258,
        "10251": 259,
        "10252": 260,
        "10256": 261,
        "10257": 262,
        "10278": 263,
        "10279": 264,
        "10280": 265,
        "10281": 266,
        "10282": 267,
        "10283": 268,
        "10284": 269,
        "10285": 270,
        "10286": 271,
        "10287": 272,
        "10288": 273,
        "10289": 274,
        "10290": 275,
        "10291": 276,
        "10292": 277,
        "10293": 278,
        "10294": 279,
        "10295": 280,
        "10296": 281,
        "10297": 282,
        "10298": 283,
        "10299": 284,
        "10300": 285,
        "10302": 286,
        "10303": 287,
        "10304": 288,
        "10305": 289,
        "10306": 290,
        "10308": 291,
        "10313": 292,
        "10314": 293,
        "10315": 294,
        "10320": 295,
        "10321": 296
    }

item_map =  {
        "0": 0, "127": 1,"134": 2,"135": 3,"161": 4,"162": 5,
        "163": 6,
        "164": 7,
        "165": 8,
        "166": 9,
        "167": 10,
        "168": 11,
        "169": 12,
        "172": 13,
        "173": 14,
        "174": 15,
        "175": 16,
        "176": 17,
        "190": 18,
        "191": 19,
        "196": 20,
        "209": 21,
        "210": 22,
        "211": 23,
        "214": 24,
        "216": 25,
        "217": 26,
        "218": 27,
        "219": 28,
        "220": 29,
        "223": 30,
        "224": 31,
        "225": 32,
        "226": 33,
        "227": 34,
        "230": 35,
        "242": 36,
        "243": 37,
        "244": 38,
        "245": 39,
        "246": 40,
        "247": 41,
        "252": 42,
        "255": 43,
        "262": 44,
        "264": 45,
        "723": 46,
        "727": 47,
        "2105": 48,
        "2177": 49
    }


def build_lut(id_map, max_id=None):
    """LUT: lut[id_originale] = indice embedding (0 se sconosciuto, noti 1..N)."""
    keys = [int(k) for k in id_map]
    size = (max_id if max_id is not None else max(keys)) + 1
    lut = np.zeros(size, dtype=np.int64)
    for k, v in id_map.items():
        lut[int(k)] = v + 1
    return lut


POKE_LUT = build_lut(pokemon_map)
MOVE_LUT = build_lut(move_map)
ABILITY_LUT = build_lut(ability_map)
ITEM_LUT = build_lut(item_map)

# num_embeddings necessari (indice 0 incluso)
N_POKE = len(pokemon_map) + 1      # 297
N_MOVE = len(move_map) + 1         # 389
N_ABILITY = len(ability_map) + 1   # 165
N_ITEM = len(item_map) + 1         # 50


def remap(lut, arr):
    """Remap vettoriale con clip: ID fuori tabella -> 0."""
    a = np.asarray(arr, dtype=np.int64)
    out = np.where((a >= 0) & (a < len(lut)), lut[np.clip(a, 0, len(lut) - 1)], 0)
    return out


In [ ]:
# ── Maschera azioni legali ───────────────────────────
"""Maschera delle azioni legali.

Spazio azioni (piatto, 360):
    dims = (s_user: 3, p_target: 2, s_target: 5, mega: 2, move: 6)
    flat  = s_user*120 + p_target*60 + s_target*12 + mega*6 + move

Convenzioni dei replay (vedi cleanstrings.py, validate con validate_mask.py):
  - pass            -> (s_user=0, trg=(0,0), mega=0, move=5)
  - switch (move=4) -> trg_slot = slot che il mon ENTRANTE aveva prima:
                       0 = mai sceso in campo, 3/4 = panchina
  - mossa m<4       -> trg_pl in {0,1}, trg_slot in {1,2}
  - move=5 con s_user>0 = mossa non riconosciuta (Struggle ecc.)
  - p_user e' sempre 0 nei dati, quindi non e' una dimensione.

Limiti noti (per questo la maschera e' permissiva, mai severa):
  - lo snapshot e' a inizio turno: non vede le dinamiche intra-turno
    (mon che agisce e poi muore, benching intra-turno dopo uno switch);
  - le mosse vengono rivelate al primo uso: una mossa nuova ha id==0
    al primo indice libero del moveset;
  - i target di mosse self/spread sono rumorosi (~5-7%).
Il Dataset forza comunque mask[azione_vera] = True come rete di sicurezza:
un -inf sull'azione vera renderebbe la loss infinita.
"""
import numpy as np  # type: ignore


class ActionMasker:
    DIMS = (3, 2, 5, 2, 6)          # s_user, p_target, s_target, mega, move
    TOTAL = 360

    def __init__(self):
        self.dims = self.DIMS
        self.total_actions = self.TOTAL

    @staticmethod
    def flat(s_user, p_target, s_target, mega, move):
        return s_user * 120 + p_target * 60 + s_target * 12 + mega * 6 + move

    @staticmethod
    def flat_batch(s_user, p_target, s_target, mega, move):
        """Versione vettoriale/tensoriale (funziona con numpy e torch)."""
        return s_user * 120 + p_target * 60 + s_target * 12 + mega * 6 + move

    def get_valid_action_mask(self, state, move, mega_available):
        """Maschera booleana [360] per un turno.

        state: dict numpy con 'player', 'slot', 'hp_ratio' (12 mon)
        move:  dict numpy con 'id' shape (12, 4)
        mega_available: True se il giocatore non ha ancora megaevoluto
                        (dalla storia delle azioni, calcolata nel Dataset).
        """
        mask = np.zeros(self.total_actions, dtype=bool)

        player = np.asarray(state['player']).reshape(12)
        slot = np.asarray(state['slot']).reshape(12)
        hp = np.asarray(state['hp_ratio']).reshape(12)
        move_id = np.asarray(move['id']).reshape(12, 4)
        own = player == 0
        own_alive = own & (hp > 0)
        any_own_active_alive = bool(np.any(own_alive & ((slot == 1) | (slot == 2))))

        # --- pass: sempre disponibile (slot vuoto / rimpiazzo / fine gioco)
        mask[self.flat(0, 0, 0, 0, 5)] = True

        megas = (0, 1) if mega_available else (0,)

        for s in (1, 2):
            # --- switch (move=4): consentito anche a slot morto (rimpiazzo)
            # verso panchina: viva allo snapshot, oppure un attivo vivo
            # potrebbe finirci durante il turno (benching intra-turno)
            for bench in (3, 4):
                if np.any(own_alive & (slot == bench)) or any_own_active_alive:
                    mask[self.flat(s, 0, bench, 0, 4)] = True
            # verso un mon mai sceso in campo (in bring-4 esistono sempre)
            if np.any(own & (slot == 0)):
                mask[self.flat(s, 0, 0, 0, 4)] = True

            # --- mosse: serve un mon proprio nello slot s.
            # Non richiediamo hp>0: se e' morto allo snapshot ha agito e
            # poi e' caduto nello stesso turno (limite dello snapshot).
            idx = np.where(own & (slot == s))[0]
            if idx.size == 0:
                continue
            i = int(idx[0])
            ids = move_id[i]
            first_free = next((j for j in range(4) if ids[j] == 0), None)
            for m in range(4):
                # id==0 e' legale solo al primo indice libero:
                # e' li' che lo scraper registra una mossa al primo uso
                if ids[m] == 0 and m != first_free:
                    continue
                for trg_pl in (0, 1):
                    for trg_sl in (1, 2):
                        for mg in megas:
                            mask[self.flat(s, trg_pl, trg_sl, mg, m)] = True
            # move=5 con s_user>0: mossa non riconosciuta (Struggle ecc.)
            for trg_pl in (0, 1):
                for trg_sl in (1, 2):
                    mask[self.flat(s, trg_pl, trg_sl, 0, 5)] = True

        return mask


In [ ]:
# ── Preprocessing numpy vettorizzato + augmentation ──
"""Preprocessing numpy vettorizzato di un file .npz (una partita).

Sostituisce i tripli loop Python di __getitem__: gli array strutturati numpy
permettono l'accesso vettoriale a tutti i campi (es. turns['pokemon']['atk']
ha shape (T, 12)). Include il remap degli ID agli indici di embedding e il
calcolo della maschera legale, cosi' il Dataset deve solo convertire a torch.

Tutto e' numpy puro: testabile senza torch.
"""
import numpy as np  # type: ignore


PREPROCESS_VERSION = 2  # bump per invalidare le cache


def preprocess_game(file_path, max_turn=49):
    """Ritorna un dict di array numpy gia' paddati a max_turn e rimappati."""
    with np.load(file_path, allow_pickle=True) as loaded:
        turns = loaded['turns']

    T = min(len(turns), max_turn)
    turns = turns[:T]
    poke = turns['pokemon']            # (T, 12) strutturato
    field = turns['field']             # (T,)

    def pad(a, fill=0):
        """Padda la prima dimensione a max_turn."""
        if a.shape[0] == max_turn:
            return np.ascontiguousarray(a)
        width = [(0, max_turn - a.shape[0])] + [(0, 0)] * (a.ndim - 1)
        return np.pad(a, width, constant_values=fill)

    # --- stato -----------------------------------------------------------
    state = {
        'id':      pad(remap(POKE_LUT, poke['poke_id'])),
        'player':  pad(poke['player'].astype(np.int64)),
        'type1':   pad(poke['type1'].astype(np.int64)),
        'type2':   pad(poke['type2'].astype(np.int64)),
        'ability': pad(remap(ABILITY_LUT, poke['ability'])),
        'item':    pad(remap(ITEM_LUT, poke['item'])),
        'slot':    pad(poke['slot'].astype(np.int64)),
        'stats': pad(np.stack([poke['hp_base'], poke['atk'], poke['def_'],
                               poke['spa'], poke['spd'], poke['spe']],
                              axis=-1).astype(np.float32) / 255.0),
        'stats_change': pad(np.stack([poke['atk_c'], poke['def_c'],
                                      poke['spa_c'], poke['spd_c'],
                                      poke['spe_c']],
                                     axis=-1).astype(np.float32) / 6.0),
        'status': pad(((poke['status_mask'][..., None].astype(np.int64)
                        >> np.arange(5, -1, -1)) & 1).astype(np.float32)),
        'hp_ratio': pad(poke['hp_ratio'].astype(np.float32)[..., None]),
    }

    # --- mosse (stack dei 4 slot mossa) ------------------------------------
    mv = [poke[f'move{m}'] for m in range(4)]          # 4 x (T, 12)
    def mstack(f): return np.stack([m[f] for m in mv], axis=-1)  # (T, 12, 4)
    raw_move_id = mstack('id')
    move = {
        'id':      pad(remap(MOVE_LUT, raw_move_id)),
        'd_class': pad(mstack('d_class').astype(np.int64)),
        't_class': pad(mstack('t_class').astype(np.int64)),
        'type':    pad(mstack('type').astype(np.int64)),
        'power':   pad(mstack('power').astype(np.float32)[..., None] / 255.0),
        'priority': pad(mstack('priority').astype(np.float32) / 8.0),
        'accuracy': pad(mstack('accuracy').astype(np.float32) / 100.0),
    }

    # --- campo -------------------------------------------------------------
    battlefield = {
        'current_weather': pad(field['weather'].astype(np.int64)),
        'speed_modifier': pad(((field['speed_mask'][..., None].astype(np.int64)
                                >> np.arange(2, -1, -1)) & 1).astype(np.float32)),
    }

    # --- azioni (stack action0/action1) -------------------------------------
    acts = [turns['action0'], turns['action1']]
    def astack(f): return np.stack([a[f].astype(np.int64) for a in acts], axis=-1)
    action = {
        'player_user':   pad(astack('usr_pl')),
        'slot_user':     pad(astack('usr_slot')),
        'player_target': pad(astack('trg_pl')),
        'slot_target':   pad(astack('trg_slot')),
        'mega':          pad(astack('mega')),
        'move':          pad(astack('move')),
    }

    # indice piatto (T, 2) nello spazio azioni 360 - stessa formula del masker
    target_flat = ActionMasker.flat_batch(
        action['slot_user'], action['player_target'],
        action['slot_target'], action['mega'], action['move'])

    # --- reward e padding ----------------------------------------------------
    # Convenzione Decision Transformer: return-to-go, cioe' l'esito finale
    # replicato su ogni turno valido (a inferenza si condiziona con 1 = vinci).
    # Con il reward solo all'ultimo turno il modello non potrebbe condizionare
    # le azioni sull'esito desiderato.
    reward = np.zeros(max_turn, dtype=np.int64)
    reward[:T] = max(0, int(field['winner'][-1]))
    padding_mask = np.zeros(max_turn, dtype=np.int64)
    padding_mask[:T] = 1

    # --- maschera legale -------------------------------------------------------
    # mega disponibile al turno t se nessuna azione precedente ha megaevoluto
    mega_any = (action['mega'][:, 0] | action['mega'][:, 1]).astype(bool)
    mega_used_before = np.concatenate([[False], np.cumsum(mega_any)[:-1] > 0])

    masker = ActionMasker()
    legal = np.ones((max_turn, masker.total_actions), dtype=bool)
    for t in range(T):
        st = {'player': state['player'][t], 'slot': state['slot'][t],
              'hp_ratio': state['hp_ratio'][t]}
        legal[t] = masker.get_valid_action_mask(
            st, {'id': raw_move_id[t]}, not mega_used_before[t])
        # rete di sicurezza: l'azione realmente giocata e' sempre legale
        for a in range(2):
            idx = target_flat[t, a]
            if 0 <= idx < masker.total_actions:
                legal[t, idx] = True

    return {
        'state': state,
        'move': move,
        'battlefield': battlefield,
        'action': action,
        'reward': reward,
        'turn': np.arange(max_turn, dtype=np.int64),
        'padding_mask': padding_mask,
        'target_flat': target_flat,
        'legal_action_mask': legal,
    }


def augment_game(g, rng, permute_rows=True, permute_moves=True):
    """Data augmentation. Ritorna una COPIA (non muta la cache del Dataset).

    1. Permuta l'ordine delle 4 mosse di ogni pokemon (una permutazione per
       mon, costante nei turni). L'indice `move` dell'azione e' solo la
       posizione nel moveset: permutarlo insegna l'invarianza. Vengono
       rimappati coerentemente l'indice mossa dell'azione target/input e le
       colonne `move` della maschera legale (con la permutazione del mon che
       agisce in quel turno/slot).
    2. Permuta le 12 righe dei pokemon nel token di stato: la posizione in
       lista e' arbitraria (l'informazione sta in `player` e `slot`), quindi
       target e maschera non cambiano.
    """
    out = {}
    for k, v in g.items():
        out[k] = ({kk: vv.copy() for kk, vv in v.items()}
                  if isinstance(v, dict) else v.copy())

    T = int(out['padding_mask'].sum())
    player = g['state']['player']   # riferimenti alle righe ORIGINALI
    slot = g['state']['slot']

    if permute_moves:
        perms = np.stack([rng.permutation(4) for _ in range(12)])  # (12, 4)
        inv = np.argsort(perms, axis=1)      # inv[i, vecchio] = nuovo indice

        # permuta gli array delle mosse: new[..., i, k] = old[..., i, perms[i, k]]
        for k, v in out['move'].items():
            idx = perms[None, :, :]
            if v.ndim == 4:                  # es. power (T, 12, 4, 1)
                idx = idx[..., None]
            out['move'][k] = np.take_along_axis(v, np.broadcast_to(idx, v.shape),
                                                axis=2)

        # rimappa l'indice mossa dell'azione (input e target)
        for t in range(T):
            for a in range(2):
                mv = int(out['action']['move'][t, a])
                s = int(out['action']['slot_user'][t, a])
                if mv < 4 and s in (1, 2):
                    rows = np.where((player[t] == 0) & (slot[t] == s))[0]
                    if rows.size:
                        out['action']['move'][t, a] = inv[int(rows[0]), mv]

        # rimappa le colonne `move` della maschera legale
        legal = out['legal_action_mask'][:T].reshape(T, 3, 2, 5, 2, 6)
        for t in range(T):
            for s in (1, 2):
                rows = np.where((player[t] == 0) & (slot[t] == s))[0]
                if rows.size:
                    p = perms[int(rows[0])]
                    legal[t, s, ..., :4] = np.take(legal[t, s], p, axis=-1)
        out['legal_action_mask'][:T] = legal.reshape(T, -1)

        # ricalcola il target piatto con l'indice mossa aggiornato
        a = out['action']
        out['target_flat'] = ActionMasker.flat_batch(
            a['slot_user'], a['player_target'], a['slot_target'],
            a['mega'], a['move'])

    if permute_rows:
        rp = rng.permutation(12)
        for k in out['state']:
            out['state'][k] = out['state'][k][:, rp]
        for k in out['move']:
            out['move'][k] = out['move'][k][:, rp]

    return out


In [ ]:
# ── Dataset ──────────────────────────────────────────
import numpy as np  # type: ignore
import torch  # type: ignore
from torch.utils.data import Dataset, DataLoader  # type: ignore



class PokemonVGCDataset(Dataset):
    """Dataset delle partite VGC.

    Ogni partita viene preprocessata UNA volta (numpy vettorizzato,
    ~2 ms/partita) e tenuta in RAM: le epoche successive costano solo
    l'indicizzazione + torch.from_numpy (zero-copy). Con la cache in RAM
    usare num_workers=0 nel DataLoader: e' gia' il percorso piu' veloce
    e evita di duplicare la cache nei worker.
    """

    def __init__(self, file_paths, max_turn=49, preload=True, verbose=False,
                 augment=False, seed=None):
        self.file_paths = list(file_paths)
        self.max_turn = max_turn
        self.augment = augment   # solo sul training set
        self._rng = np.random.default_rng(seed)
        self._cache = [None] * len(self.file_paths)
        if preload:
            for i in range(len(self.file_paths)):
                self._cache[i] = preprocess_game(self.file_paths[i], max_turn)
                if verbose and (i + 1) % 200 == 0:
                    print(f'  preprocess {i + 1}/{len(self.file_paths)}')

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        g = self._cache[idx]
        if g is None:
            g = self._cache[idx] = preprocess_game(self.file_paths[idx],
                                                   self.max_turn)
        if self.augment:
            g = augment_game(g, self._rng)
        t = torch.from_numpy
        return {
            'state': {k: t(v) for k, v in g['state'].items()},
            'move': {k: t(v) for k, v in g['move'].items()},
            'battlefield': {k: t(v) for k, v in g['battlefield'].items()},
            'action': {k: t(v) for k, v in g['action'].items()},
            'reward': t(g['reward']),
            'turn': t(g['turn']),
            'padding_mask': t(g['padding_mask']),
            'target_flat': t(g['target_flat']),          # (max_turn, 2)
            'legal_action_mask': t(g['legal_action_mask']),  # (max_turn, 360) bool
        }


In [ ]:
# ── Embedding ────────────────────────────────────────
"""Embedding di stato, azione, ricompensa e turno.

Gli ID (pokemon, mossa, ability, item) arrivano GIA' rimappati agli indici
di embedding dal Dataset (vedi id_maps.py / preprocess.py): indice 0 =
sconosciuto/padding. Qui non si fa nessun lookup di dizionari.
"""
import torch  # type: ignore
import torch.nn as nn  # type: ignore



class Embedding(nn.Module):
    def __init__(self, d_model=256, feat_dim=16, max_turn=49):
        super().__init__()
        self.d_model = d_model

        # --- stato: 12 pokemon, feature discrete
        self.embed_player = nn.Embedding(2, feat_dim)
        self.embed_id = nn.Embedding(N_POKE, feat_dim)        # 242
        self.embed_type1 = nn.Embedding(19, feat_dim)
        self.embed_type2 = nn.Embedding(19, feat_dim)
        self.embed_ability = nn.Embedding(N_ABILITY, feat_dim)  # 150
        self.embed_item = nn.Embedding(N_ITEM, feat_dim)        # 40
        self.embed_slot = nn.Embedding(5, feat_dim)

        # --- stato: feature continue
        self.embed_stats = nn.Linear(6, feat_dim)
        self.embed_stats_change = nn.Linear(5, feat_dim)
        self.embed_status = nn.Linear(6, feat_dim)
        self.embed_hp_ratio = nn.Linear(1, feat_dim)

        # --- mosse (12 pokemon x 4 mosse)
        self.embed_id_move = nn.Embedding(N_MOVE, feat_dim)   # 323
        self.embed_d_class = nn.Embedding(4, feat_dim)        # 0..3
        self.embed_move_type = nn.Embedding(19, feat_dim)
        self.embed_t_class = nn.Embedding(17, feat_dim)       # 0..16
        self.embed_power = nn.Linear(1, feat_dim)
        self.embed_priority = nn.Linear(1, feat_dim)
        self.embed_accuracy = nn.Linear(1, feat_dim)

        # --- campo
        self.embed_current_weather = nn.Embedding(5, feat_dim)
        self.embed_speed_modifier = nn.Linear(3, feat_dim)

        # dimensione dello stato completo: 12 pokemon x 11 feature
        # + 12x4 mosse x 7 feature + 2 feature di campo
        state_in = (12 * 11 + 12 * 4 * 7 + 2) * feat_dim   # 7520 con feat_dim=16
        self.state_proj = nn.Linear(state_in, d_model)

        # --- azione (2 azioni per turno x 6 componenti)
        self.embed_player_user = nn.Embedding(2, feat_dim)
        self.embed_slot_user = nn.Embedding(3, feat_dim)      # 0=pass, 1, 2
        self.embed_player_target = nn.Embedding(2, feat_dim)
        self.embed_slot_target = nn.Embedding(5, feat_dim)
        self.embed_mega = nn.Embedding(2, feat_dim)
        self.embed_move = nn.Embedding(6, feat_dim)
        self.action_proj = nn.Linear(2 * 6 * feat_dim, d_model)

        # --- ricompensa (return-to-go 0/1) e turno
        self.embed_reward = nn.Embedding(2, d_model)
        self.embed_turn = nn.Embedding(max_turn, d_model)

    def forward(self, state, move, battlefield, action, reward, turn,
                padding_mask=None):
        batch_size, seq_len = state['id'].shape[:2]

        # --- stato
        pokemon_emb = torch.cat([
            self.embed_player(state['player']),
            self.embed_id(state['id']),
            self.embed_type1(state['type1']),
            self.embed_type2(state['type2']),
            self.embed_ability(state['ability']),
            self.embed_item(state['item']),
            self.embed_slot(state['slot']),
            self.embed_stats(state['stats']),
            self.embed_stats_change(state['stats_change']),
            self.embed_status(state['status']),
            self.embed_hp_ratio(state['hp_ratio']),
        ], dim=-1)                                   # (B, T, 12, 11*feat)
        pokemon_flat = pokemon_emb.flatten(2)        # (B, T, 12*11*feat)

        move_emb = torch.cat([
            self.embed_id_move(move['id']),
            self.embed_move_type(move['type']),
            self.embed_d_class(move['d_class']),
            self.embed_t_class(move['t_class']),
            self.embed_power(move['power']),
            self.embed_priority(move['priority'].unsqueeze(-1)),
            self.embed_accuracy(move['accuracy'].unsqueeze(-1)),
        ], dim=-1)                                   # (B, T, 12, 4, 7*feat)
        move_flat = move_emb.flatten(2)              # (B, T, 12*4*7*feat)

        weather_emb = self.embed_current_weather(battlefield['current_weather'])
        speed_emb = self.embed_speed_modifier(battlefield['speed_modifier'])

        full_state = torch.cat([pokemon_flat, move_flat,
                                weather_emb, speed_emb], dim=-1)
        state_emb = self.state_proj(full_state)      # (B, T, d_model)

        # --- azione (slot_user usato direttamente: 0 = pass)
        full_action = torch.cat([
            self.embed_player_user(action['player_user']),
            self.embed_slot_user(action['slot_user']),
            self.embed_player_target(action['player_target']),
            self.embed_slot_target(action['slot_target']),
            self.embed_mega(action['mega']),
            self.embed_move(action['move']),
        ], dim=-1)                                   # (B, T, 2, 6*feat)
        action_emb = self.action_proj(full_action.flatten(2))

        # --- ricompensa e turno
        reward_emb = self.embed_reward(reward)
        turn_emb = self.embed_turn(turn)

        state_emb = state_emb + turn_emb
        action_emb = action_emb + turn_emb
        reward_emb = reward_emb + turn_emb

        # interleaving (R_t, s_t, a_t): (B, 3T, d_model)
        final_inputs = torch.stack(
            [reward_emb, state_emb, action_emb],
            dim=2).reshape(batch_size, 3 * seq_len, self.d_model)

        stacked_padding_mask = (padding_mask.repeat_interleave(3, dim=1)
                                if padding_mask is not None else None)
        return final_inputs, stacked_padding_mask


In [ ]:
# ── Self-Attention / Transformer Block ───────────────
"""Self-Attention e Transformer Block.

Usa F.scaled_dot_product_attention (Flash/Memory-efficient attention dove
disponibile): molto piu' veloce e con meno memoria della versione manuale.

Fix importante: nella versione precedente le righe di query dei turni di
padding avevano TUTTI gli score a -inf -> softmax = NaN, e i NaN si
propagavano a tutta la sequenza dal secondo blocco in poi (0 * NaN = NaN
nella matmul dei value). Ora la diagonale resta sempre attendibile, cosi'
nessuna riga e' completamente mascherata.
"""
import torch  # type: ignore
import torch.nn as nn  # type: ignore
import torch.nn.functional as F  # type: ignore


class SelfAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=8, max_seq_length=147):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.query = nn.Linear(d_model, d_model, bias=False)
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.unifyheads = nn.Linear(d_model, d_model, bias=False)

        causal = torch.tril(torch.ones(max_seq_length, max_seq_length,
                                       dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, padding_mask=None):
        B, T, D = x.size()

        def split(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = split(self.query(x)), split(self.key(x)), split(self.value(x))

        # maschera combinata: True = posizione attendibile
        attn_mask = self.causal_mask[:T, :T]              # (T, T)
        if padding_mask is not None:
            keep = padding_mask.bool()                     # (B, T)
            attn_mask = attn_mask.unsqueeze(0) & keep.unsqueeze(1)  # (B, T, T)
            # la diagonale resta sempre attendibile: evita righe tutte-False
            # (query di padding) che producono NaN nel softmax
            eye = torch.eye(T, dtype=torch.bool, device=x.device)
            attn_mask = attn_mask | eye.unsqueeze(0)
            attn_mask = attn_mask.unsqueeze(1)             # (B, 1, T, T)

        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.unifyheads(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=8, dropout=0.0, max_seq_length=147):
        super().__init__()
        self.self_attention = SelfAttention(d_model, n_heads, max_seq_length)
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        x = x + self.dropout(self.self_attention(self.layer_norm1(x),
                                                 padding_mask))
        x = x + self.feed_forward(self.layer_norm2(x))
        return x


In [ ]:
# ── Decision Transformer ─────────────────────────────
#Decision Transformer

import torch # type: ignore
import torch.nn as nn # type: ignore
import torch.nn.functional as F # type: ignore



class DecisionTransformer(nn.Module):
    def __init__(self, action_dim=360, d_model=256, n_heads=8, depth=6, max_turn=49, dropout=0.1):
        super().__init__()
        self.action_dim = action_dim #tutte le possibili azioni (mosse)
        self.d_model=d_model
        self.seq_length=max_turn
        max_seq_length=3*max_turn #max_seq_length is 3 times the number of tokens (R,s,a) for each turn

        #Embedding layer for states, actions, and rewards
        self.token_embedding = Embedding(d_model=d_model, max_turn=max_turn)

        # Transformer blocks
        self.tblocks=nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                dropout=dropout,
                max_seq_length=max_seq_length
                ) for _ in range(depth)
        ])
        
        self.predict_action=nn.Linear(d_model, 2*self.action_dim) #per predire il token dell'azione
    
    def forward(self, state, move, battlefield, action, reward, turn, padding_mask=None, legal_action_mask=None):
        #x is a tensor of shape (batch_size, seq_length, d_model)
        batch_size=state['id'].size(0)

        x, stacked_padding_mask=self.token_embedding(state, move, battlefield, action, reward, turn, padding_mask) #embedding layer
        
        for block in self.tblocks:
            x=block(x, padding_mask=stacked_padding_mask) #transformer blocks

        
        x=x.reshape(batch_size, self.seq_length, 3, self.d_model).permute(0,2,1,3) #reshape to (batch_size, 3, seq_length, d_model)

        state_representation=x[:,1] #prendo solo la componente di stato di x. ha dimensione (batch_size, seq_length, d_modele) 
        
        action_preds=self.predict_action(state_representation) #linear layer to get logits for each action
        #va fatta una maschera per le azioni illegali da applicare ad action_preds la maschera mette le azioni illegali di action_preds a -inf
        action_preds=action_preds.view(batch_size, self.seq_length, 2, self.action_dim) 

        if legal_action_mask is not None:
            # legal_action_mask: bool, shape (batch_size, seq_length, action_dim)
            # True = azione lecita. Le due teste condividono lo stesso spazio -> stesso mask per entrambe
            mask = legal_action_mask.unsqueeze(2)  # (batch, seq, 1, action_dim) -> broadcast su dim=2 (le due teste)
            action_preds = action_preds.masked_fill(~mask, float('-inf'))
            
        return F.log_softmax(action_preds, dim=-1) #log softmax over the last dimension (num_tokens)


class AmpDecisionTransformer(DecisionTransformer):
    """Autocast DENTRO il forward: con nn.DataParallel ogni replica gira in un
    thread separato e il contesto autocast esterno non si propaga."""
    def forward(self, *args, **kwargs):
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            return super().forward(*args, **kwargs)


In [ ]:
# ── Training loop ────────────────────────────────────
def _base(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def save_checkpoint(save_dir, epoch, model, optimizer, loss,
                    filename="latest_checkpoint.pth"):
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, filename)
    torch.save({'epoch': epoch,
                'model_state_dict': _base(model).state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss}, path)
    return path


def load_checkpoint(filepath, model, optimizer):
    if filepath and os.path.exists(filepath):
        ckpt = torch.load(filepath, map_location='cpu')
        try:
            _base(model).load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        except RuntimeError as e:
            print(f'Checkpoint incompatibile con questa configurazione, '
                  f'riparto da zero. ({str(e)[:120]}...)')
            return 0
        print(f"Checkpoint caricato: riprendo dall'epoca {ckpt['epoch'] + 1} "
              f"(loss {ckpt['loss']:.4f})")
        return ckpt['epoch'] + 1
    print("Nessun checkpoint trovato. Inizio da zero.")
    return 0


def _to_device(batch, device, non_blocking):
    out = {}
    for k, v in batch.items():
        if isinstance(v, dict):
            out[k] = {kk: vv.to(device, non_blocking=non_blocking)
                      for kk, vv in v.items()}
        else:
            out[k] = v.to(device, non_blocking=non_blocking)
    return out


def _run_batch(model, batch, criterion, use_mask=True):
    """Ritorna (loss, n_azioni_valide, n_azioni_corrette)."""
    legal = batch['legal_action_mask'] if use_mask else None
    log_probs = model(batch['state'], batch['move'], batch['battlefield'],
                      batch['action'], batch['reward'], batch['turn'],
                      batch['padding_mask'], legal_action_mask=legal)
    # NB: appiattito a 2D invece di permute(0,3,1,2): il kernel CUDA di
    # nll_loss2d richiede input contiguo ("grad_input must be contiguous")
    target = batch['target_flat']
    loss_el = criterion(log_probs.reshape(-1, log_probs.size(-1)),
                        target.reshape(-1)).view_as(target)      # (B, T, 2)

    mask = batch['padding_mask'].unsqueeze(-1).expand_as(loss_el).float()
    loss = (loss_el * mask).sum() / mask.sum()

    with torch.no_grad():
        pred = log_probs.argmax(dim=-1)
        correct = ((pred == target).float() * mask).sum()
    return loss, mask.sum(), correct


def evaluate(model, dataloader, criterion, device, non_blocking, use_amp,
             use_legal_mask=True):
    model.eval()
    tl = tn = tc = 0.0
    with torch.no_grad():
        for batch in dataloader:
            batch = _to_device(batch, device, non_blocking)
            with torch.amp.autocast('cuda', enabled=use_amp):
                loss, n, correct = _run_batch(model, batch, criterion,
                                              use_legal_mask)
            tl += loss.item() * n.item()
            tn += n.item()
            tc += correct.item()
    return tl / max(tn, 1), tc / max(tn, 1)


def set_backbone_frozen(model, frozen):
    """Congela embedding + tutti i transformer block TRANNE l'ultimo.
    Restano sempre allenabili: ultimo blocco e testa predict_action."""
    base = _base(model)
    for p in base.token_embedding.parameters():
        p.requires_grad = not frozen
    for blk in base.tblocks[:-1]:
        for p in blk.parameters():
            p.requires_grad = not frozen


def train_decision_transformer(model, dataloader_training,
                               dataloader_validation, num_epochs, device,
                               lr=1e-4, save_dir='checkpoints',
                               resume_from=None, use_legal_mask=True,
                               warmup_epochs=0, freeze_epochs=0,
                               eval_first=False, patience=4):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)

    # warmup lineare (se richiesto) + cosine decay fino al 5% del LR
    def lr_lambda(epoch):
        if warmup_epochs and epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        t = (epoch - warmup_epochs) / max(1, num_epochs - warmup_epochs)
        return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(t, 1.0)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler(enabled=use_amp)
    non_blocking = device.type == 'cuda'
    criterion = nn.NLLLoss(reduction='none')

    start_epoch = load_checkpoint(resume_from, model, optimizer) \
        if resume_from else 0
    for _ in range(start_epoch):
        scheduler.step()          # riallinea lo schedule dopo un resume
    best_val_loss = float('inf')
    epochs_no_improve = 0
    if eval_first:
        vl, va = evaluate(model, dataloader_validation, criterion, device,
                          non_blocking, use_amp, use_legal_mask)
        print(f'Baseline zero-shot | val loss {vl:.4f} acc {va:.3f}')

    frozen = False
    for epoch in range(start_epoch, num_epochs):
        # ---- layer freezing (solo fine-tuning) ----
        if freeze_epochs and epoch < freeze_epochs and not frozen:
            set_backbone_frozen(model, True)
            frozen = True
            print(f'Backbone congelato per le prime {freeze_epochs} epoche '
                  f'(si allenano ultimo blocco + testa)')
        elif frozen and epoch >= freeze_epochs:
            set_backbone_frozen(model, False)
            frozen = False
            print('Backbone scongelato: fine-tuning completo')

        # ---- training ----
        model.train()
        t0 = time.time()
        tot_loss = tot_n = tot_correct = 0.0
        for batch in dataloader_training:
            batch = _to_device(batch, device, non_blocking)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                loss, n, correct = _run_batch(model, batch, criterion,
                                              use_legal_mask)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            tot_loss += loss.item() * n.item()
            tot_n += n.item()
            tot_correct += correct.item()
        train_loss = tot_loss / tot_n
        train_acc = tot_correct / tot_n

        # ---- validation ----
        val_loss, val_acc = evaluate(model, dataloader_validation, criterion,
                                     device, non_blocking, use_amp,
                                     use_legal_mask)

        scheduler.step()
        dt = time.time() - t0
        print(f"Epoch {epoch + 1:3d}/{num_epochs} | "
              f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.3f} | "
              f"lr {optimizer.param_groups[0]['lr']:.1e} | {dt:.1f}s")

        save_checkpoint(save_dir, epoch, model, optimizer, train_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            save_checkpoint(save_dir, epoch, model, optimizer, val_loss,
                            "best_model.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'Early stopping: val loss ferma a {best_val_loss:.4f} '
                      f'da {patience} epoche (epoca {epoch + 1})')
                break

    return model



In [ ]:
# ── Main: 2 GPU con nn.DataParallel ──────────────────
def find_data():
    for pat in DATA_GLOBS:
        fs = sorted(glob.glob(pat, recursive=True))
        if fs:
            print(f'{len(fs)} partite da {pat}')
            return fs
    raise FileNotFoundError(f'nessun .npz trovato in {DATA_GLOBS}')

def run():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device} | GPU disponibili: {torch.cuda.device_count()} | '
          f'modalita: {"FINE-TUNING" if FINETUNE else "addestramento normale"}')

    files = find_data()
    random.Random(42).shuffle(files)
    if FAST:
        files = files[:300]
    n = len(files)
    n_val, n_test = max(1, n // 10), max(1, n // 10)
    train_files = files[:n - n_val - n_test]
    val_files = files[n - n_val - n_test:n - n_test]
    test_files = files[n - n_test:]
    print(f'{len(train_files)} train / {len(val_files)} val / {len(test_files)} test')

    t0 = time.time()
    ds_train = PokemonVGCDataset(train_files, max_turn=MAX_TURN, verbose=True,
                                 augment=AUGMENT)
    ds_val = PokemonVGCDataset(val_files, max_turn=MAX_TURN)
    print(f'Preprocessing: {time.time() - t0:.1f}s (una tantum, poi tutto in RAM)')

    dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH, shuffle=False,
                        num_workers=0, pin_memory=True)

    model = AmpDecisionTransformer(action_dim=360, d_model=D_MODEL,
                                   n_heads=N_HEADS, depth=DEPTH,
                                   max_turn=MAX_TURN, dropout=DROPOUT)
    print(f'Parametri: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

    if FINETUNE:
        sd = torch.load(PRETRAINED, map_location='cpu')
        if 'model_state_dict' in sd:      # accetta sia state_dict sia checkpoint
            sd = sd['model_state_dict']
        model.load_state_dict(sd)
        print(f'Pesi pre-addestrati caricati: {PRETRAINED}')

    if torch.cuda.device_count() > 1:
        print(f'Uso {torch.cuda.device_count()} GPU (DataParallel)')
        model = nn.DataParallel(model)

    resume = os.path.join(SAVE_DIR, 'latest_checkpoint.pth')
    train_decision_transformer(
        model, dl_train, dl_val, num_epochs=EPOCHS, device=device, lr=LR,
        save_dir=SAVE_DIR, resume_from=resume if os.path.exists(resume) else None,
        warmup_epochs=WARMUP_EPOCHS, freeze_epochs=FREEZE_EPOCHS,
        eval_first=FINETUNE)

    base = model.module if isinstance(model, nn.DataParallel) else model
    name = 'vgc_decision_transformer_ft.pth' if FINETUNE else 'vgc_decision_transformer.pth'
    out = os.path.join(os.path.dirname(SAVE_DIR) or '.', name)
    torch.save(base.state_dict(), out)
    print(f'Modello salvato: {out}')

run()
